## 🔧 환경 세팅 (최초 1회만)

이 노트북은 **`redteam`** conda env에서 실행됩니다.

> **단일 env 통합**: garak 0.15.0부터 `openai>=2.0`을 지원하면서 PyRIT와 의존성 충돌이 사라졌습니다.
> `1_garak`, `2_pyrit`, `3_promptfoo` 모두 동일한 **`redteam`** env에서 실행합니다.

### 0) Miniconda 설치 (Python 미설치 OK)

> **Python을 따로 설치할 필요 없습니다.** Miniconda가 자체 Python을 함께 설치해줍니다.

터미널에서 `conda --version`이 동작하면 이 단계는 건너뛰세요.

**macOS (Apple Silicon, M1/M2/M3)**
```bash
# Homebrew 사용 (권장)
brew install --cask miniconda

# 또는 공식 인스톨러
curl -O https://repo.anaconda.com/miniconda/Miniconda3-latest-MacOSX-arm64.sh
bash Miniconda3-latest-MacOSX-arm64.sh
```

**macOS (Intel)**
```bash
curl -O https://repo.anaconda.com/miniconda/Miniconda3-latest-MacOSX-x86_64.sh
bash Miniconda3-latest-MacOSX-x86_64.sh
```

**Windows**
- [Miniconda 공식 페이지](https://docs.conda.io/projects/miniconda/en/latest/) 에서 `.exe` 다운로드 후 실행
- 설치 후 **Anaconda Prompt** 를 열어 이후 명령 실행

**Linux**
```bash
wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
bash Miniconda3-latest-Linux-x86_64.sh
```

설치 후 새 터미널을 열고 확인:
```bash
conda --version    # ex) conda 24.x.x
python --version   # ex) Python 3.12.x  (conda base env의 Python)
```

### 1) conda env 생성 (터미널에서 1회만)

```bash
# 방법 A: 스크립트 한 줄
bash setup.sh

# 방법 B: 수동 명령
conda create -n redteam python=3.11 -y
conda run -n redteam pip install ipykernel
conda run -n redteam python -m ipykernel install --user --name redteam --display-name "redteam"
```

> conda 대신 venv·uv 등을 쓰는 경우 `setup.sh`를 본인 환경에 맞게 수정하세요.

### 2) VSCode에서 커널 선택

이 노트북을 열고 우상단 **Select Kernel** → **`redteam`** 선택.

(`1_garak_tutorial_en.ipynb` / `2_pyrit_tutorial.ipynb` 도 동일하게 `redteam` env에서 실행)

### 3) 나머지 의존성

아래 셀이 자동으로 `redteam` env에 promptfoo 의존성을 설치합니다 — **수동 작업 불필요**.


In [1]:
# ════════════════════════════════════════════════════════════════════
# env 검증 — Python / conda / 현재 커널이 기대한 env인지 확인
#   잘못된 env이면 의존성 충돌이 발생하므로 즉시 경고
# ════════════════════════════════════════════════════════════════════
import os, sys, shutil, subprocess

EXPECTED_ENV = "redteam"
MIN_PY = (3, 10)
MAX_PY = (3, 12)

# (1) Python 버전 검증
py = sys.version_info
py_ok = MIN_PY <= (py.major, py.minor) <= MAX_PY
py_mark = "✅" if py_ok else "⚠️ "
print(f"{py_mark} Python       : {py.major}.{py.minor}.{py.micro}  (권장 {MIN_PY[0]}.{MIN_PY[1]} ~ {MAX_PY[0]}.{MAX_PY[1]})")
if not py_ok:
    print(f"   → 권장 범위를 벗어났습니다. 'conda create -n redteam python=3.11 -y' 로 재생성 권장")

# (2) conda 명령 존재 확인
conda_path = shutil.which("conda")
if conda_path:
    try:
        ver = subprocess.run(["conda", "--version"], capture_output=True, text=True, timeout=5).stdout.strip()
    except Exception:
        ver = "(버전 조회 실패)"
    print(f"✅ conda        : {ver}  ({conda_path})")
else:
    print("⚠️  conda 명령을 찾지 못했습니다.")
    print("   → 위 markdown '0) Miniconda 설치' 단계를 먼저 진행하세요.")
    print("   → 이미 설치했다면 새 터미널/VSCode 재시작 후 'conda --version' 으로 확인.")

# (3) 현재 활성화된 conda env 검증
current_env = os.environ.get("CONDA_DEFAULT_ENV", "")
if current_env != EXPECTED_ENV:
    print(f"⚠️  현재 conda env: {current_env!r}")
    print(f"   예상 env      : {EXPECTED_ENV!r}")
    print(f"   → VSCode 우상단 'Select Kernel'에서 '{EXPECTED_ENV}' 선택 후 다시 실행하세요.")
    print(f"   → env가 없다면 위 markdown의 'conda env 생성' 명령을 먼저 실행하세요.")
else:
    print(f"✅ conda env    : {current_env}")
    print(f"✅ kernel python: {sys.executable}")


✅ Python       : 3.11.15  (권장 3.10 ~ 3.12)
✅ conda        : conda 25.11.0  (/opt/anaconda3/condabin/conda)
✅ conda env    : redteam
✅ kernel python: /opt/anaconda3/envs/redteam/bin/python


# Promptfoo Redteam Init 웹 UI 실행 튜토리얼 

이 노트북은 promptfoo의 **redteam init 웹 UI**를 띄우는 절차를 다룹니다.

### 특징
- **`[P5-1] Red-Teaming Framework` 폴더 안에서 독립 실행** — 외부 의존 X
- **Node.js 24.14.1을 workspace에 자동 격리 설치** (`nodeenv` 사용, sudo 불필요)
- **promptfoo 0.121.11을 workspace에 자동 격리 설치** (`npm install`)
- macOS / Linux / Windows 모두 호환
- API 키는 **프로젝트 폴더의 `.env`에서 자동 로드** (`OPENAI_API_KEY=sk-...`)

### 진행 순서
1. **환경 설정** — Node + promptfoo 격리 설치 + `.env` 로드 (아래 첫 코드 셀, 자동)
2. **promptfoo 웹 UI 실행** — `!promptfoo redteam init` 한 줄 (`2) Promptfoo Redteam Init 웹 UI 실행` 섹션)
3. **promptfoo의 한계** — 사용 전 알아둘 도구 자체의 제약 (`3) Promptfoo의 한계` 섹션)

### 전제 조건
- **Python 3.10+** (이 노트북을 실행 중이라면 OK)
- **인터넷** (Node + promptfoo 다운로드용)
- **OpenAI API 키**

In [2]:
import os
import sys
import importlib.util
from pathlib import Path

# 버전 고정
NODE_VERSION = "24.14.1"
PROMPTFOO_VERSION = "0.121.11"

# 1) 작업 디렉토리
WORKSPACE = Path.cwd() / "promptfoo_workspace"
WORKSPACE.mkdir(exist_ok=True)
os.chdir(WORKSPACE)

# 2) nodeenv 설치 (이미 있으면 건너뜀)
if importlib.util.find_spec("nodeenv") is None:
    !pip install nodeenv

# 3) Node 격리 설치 (디렉토리명에 버전 포함 → 버전 바뀌면 자동 재설치)
NODE_DIR = WORKSPACE / f".node-{NODE_VERSION}"
NODE_BIN = NODE_DIR / ("Scripts" if os.name == "nt" else "bin")
node_exe = NODE_BIN / ("node.exe" if os.name == "nt" else "node")

if not node_exe.exists():
    print(f"⏳ Node {NODE_VERSION} 다운로드 중 (1~2분)...")
    !python -m nodeenv --node={NODE_VERSION} --prebuilt "{NODE_DIR}"

# 4) PATH에 Node + promptfoo 로컬 bin 등록
PROMPTFOO_BIN = WORKSPACE / "node_modules" / ".bin"
os.environ["PATH"] = f"{NODE_BIN}{os.pathsep}{PROMPTFOO_BIN}{os.pathsep}{os.environ.get('PATH', '')}"

# 5) promptfoo 설치 (이미 있으면 건너뜀)
if not (WORKSPACE / "node_modules" / "promptfoo" / "package.json").exists():
    print(f"⏳ promptfoo@{PROMPTFOO_VERSION} 설치 중 (1~2분)...")
    !npm install --silent promptfoo@{PROMPTFOO_VERSION}

# 6) promptfoo 첫 실행 시 멈춤 방지
os.environ["PROMPTFOO_DISABLE_TELEMETRY"] = "1"
os.environ["PROMPTFOO_DISABLE_UPDATE"] = "1"

# 7) OpenAI API 키 — .env 파일에서 로드
# python-dotenv (이미 import 가능하면 건너뜀)
import importlib.util as _ilu
if _ilu.find_spec("dotenv") is None:
    !pip install python-dotenv
from dotenv import load_dotenv

# .env 자동 탐색: 노트북 옆 → 상위 디렉토리
_env_search = [Path.cwd() / ".env", Path.cwd().parent / ".env"]
_env_loaded = next((p for p in _env_search if p.exists()), None)
if _env_loaded is None:
    _tried = "\n  - ".join(str(p) for p in _env_search)
    raise FileNotFoundError(
        ".env 파일을 찾지 못했습니다. 다음 위치 중 하나에 만들고 "
        "`OPENAI_API_KEY=sk-...` 형식으로 작성하세요:\n  - " + _tried
    )

load_dotenv(_env_loaded)
if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError(f"{_env_loaded} 에 OPENAI_API_KEY가 비어 있습니다.")
print(f"✅ .env 로드: {_env_loaded}")

import subprocess
node_ver = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip()
print(f"✅ workspace : {WORKSPACE}")
print(f"✅ Node      : {node_ver}")
print(f"✅ promptfoo : {PROMPTFOO_VERSION} + OPENAI_API_KEY 준비 완료")

  Using cached nodeenv-1.10.0-py2.py3-none-any.whl.metadata (24 kB)
Using cached nodeenv-1.10.0-py2.py3-none-any.whl (23 kB)
⏳ Node 24.14.1 다운로드 중 (1~2분)...
 * Install prebuilt node (24.14.1) ..... done.
⏳ promptfoo@0.121.11 설치 중 (1~2분)...
✅ .env 로드: /Users/selectstar/P5-1_Red-Teaming-Framework/.env
✅ workspace : /Users/selectstar/P5-1_Red-Teaming-Framework/promptfoo_workspace
✅ Node      : v24.14.1
✅ promptfoo : 0.121.11 + OPENAI_API_KEY 준비 완료


## 2) Promptfoo Redteam Init 웹 UI 실행

위 환경 설정 셀에서 promptfoo를 workspace에 격리 설치하고 PATH를 잡아 두었으므로, **`!promptfoo redteam init` 한 줄**로 실행됩니다.

- 웹 UI: **http://localhost:15500/redteam/setup**
- 셀이 "실행 중" 상태로 머무릅니다 — 서버 프로세스가 살아있어야 UI가 동작

### 사용 흐름
1. 아래 셀 실행 → 출력의 URL을 브라우저로 열기
2. UI에서 타겟 / 공격 플러그인 / 전략 등 설정
3. 설정 완료 시 `redteam.yaml`이 `promptfoo_workspace/`에 저장됨
4. 셀 정지(■ Stop) 버튼으로 서버 종료

### 원격 서버에서 실행 중이라면
```bash
ssh -L 15500:localhost:15500 <user>@<server>
# 로컬 브라우저에서 http://localhost:15500/redteam/setup
```

In [ ]:
# 위 환경 설정 셀에서 PATH를 잡아 두었으므로 `promptfoo` 명령 직접 호출 가능
# Jupyter ! 매직이 os.environ을 상속 → OPENAI_API_KEY 자동 전달
# 셀이 실행 중인 동안 서버가 살아 있음 → 사용 끝나면 ■ Stop으로 중단

# 첫 실행은 이메일 등록이 필요하여 터미널에서 진행

!promptfoo redteam init

Server running at http://localhost:15500 and monitoring for new evals.
Press Ctrl+C to stop the server
^C


## 3) Promptfoo의 한계

본 튜토리얼이 아닌, **promptfoo 도구 자체**가 안고 있는 제약입니다. 사용 전 미리 알아두면 결과 해석에 도움이 됩니다.

1. **웹 UI 중심 워크플로**: `redteam init`은 인터랙티브 웹 UI에 의존합니다. CI/CD나 headless 환경에서는 `redteam.yaml`을 직접 작성하고 `promptfoo eval`로 비교 실행하는 별도 흐름이 필요합니다.
2. **LLM-as-judge 의존성**: 응답 평가가 또 다른 LLM(judge) 호출에 의존하므로, judge 자체의 비결정성·편향이 결과에 직접 반영됩니다. 사람 라벨러보다 합의도가 낮을 수 있습니다.
3. **공격 플러그인 다양성의 상한**: 사전 정의된 redteam 플러그인(harmful, jailbreak, PII 등) 조합에 한정됩니다. 실제 adversary의 창의성·적응성은 시뮬레이션되지 않습니다.
4. **데이터셋 정적성**: 내장 시드·플러그인은 시점 데이터입니다. 시간이 지나면 모델 학습 데이터에 노출되어 약발이 떨어지거나, 사회/안전 가이드라인과 어긋날 수 있습니다.
5. **언어 편향**: 영어 중심으로 설계되어, 한국어 등 비영어 환경에서는 일부 플러그인/judge의 품질이 저하될 수 있습니다.
6. **Provider 커버리지**: OpenAI/Anthropic/Google 등 주요 API는 지원하지만, 사내 모델·온프레미스 LLM·특수 형태(Streaming-only 등)는 별도 어댑터 작성이 필요합니다.
7. **API 비용 누적**: 1회 redteam eval이 수십~수백 케이스 × (타겟 + judge) 호출을 발생시킵니다. 비용을 사전 추산하지 않으면 빠르게 누적됩니다.
8. **재현성**: LLM 응답은 비결정적이라 동일 설정에서도 결과가 달라집니다. 단발성 실행보다는 반복 실행 후 통계적 분석이 필요합니다.
9. **버전 의존성**: promptfoo CLI는 빈번히 업데이트되며 일부 옵션/스키마가 바뀝니다. 본 노트북은 `@0.121.11`로 버전 고정해 두었으나, 최신 기능이 필요하면 `@latest`로 바꿔야 합니다.
10. **"공격 성공 ≠ 실제 위해"**: 모델이 거절하지 않고 답했다고 해서 그 응답이 실제 위해 가능 정보라는 보장은 없습니다. 정성 분석(human-in-the-loop)이 필수입니다.
11. **발견 도구, 방어 도구 아님**: promptfoo는 *발견*에 특화된 평가 도구이지 *방어* 도구가 아닙니다. 발견된 취약점의 완화는 가드레일/시스템 프롬프트/파인튜닝 등 다른 스택의 역할입니다.